# Fin-GAN: Forecasting and Classifying Financial Time Series via GANs

**Paper**: Vuletić, Prenzel & Cucuringu (2024), *Quantitative Finance*, 24:2, 175-199

---

## 1. Paper Summary

### Core Idea
The paper introduces **Fin-GAN**, a GAN-based framework for **probabilistic forecasting** of financial time series. Unlike traditional point-estimate models (LSTM, ARIMA), Fin-GAN produces full conditional probability distributions of future returns, enabling uncertainty quantification and a weighted trading strategy.

### Architecture: ForGAN + Novel Loss
Fin-GAN builds on the **ForGAN** architecture (Koochali et al. 2019) — a conditional GAN where:
- The **generator** receives Gaussian noise + a condition window of L past returns, both processed through LSTM layers, and outputs a predicted next-step return.
- The **discriminator** receives either (condition + real target) or (condition + generated target), processed through its own LSTM, and outputs a probability of the input being real.

The key contribution is a novel **economics-driven loss function** for the generator:

$$\mathcal{L}_G = J^{(G)}_{BCE} - \alpha \cdot PnL^* + \beta \cdot MSE - \gamma \cdot SR^* + \delta \cdot STD$$

where:
- $J^{(G)}_{BCE}$: standard GAN generator loss (BCE)
- $PnL^* = \frac{1}{n}\sum_i \tanh(k \hat{x}_i) x_i$: smooth PnL approximation (sign → tanh)
- $MSE$: mean squared error between forecast and target
- $SR^*$: differentiable Sharpe ratio of approximate PnLs within the minibatch
- $STD$: standard deviation of approximate PnLs

Hyperparameters $\alpha, \beta, \gamma, \delta$ are set via **gradient norm matching** — no manual tuning.

### Pros
- Produces **distributional forecasts** with uncertainty estimates, enabling a weighted trading strategy
- Economics-driven loss **shifts generated distributions** in the profitable direction
- **Alleviates mode collapse** — the PnL term helps the generator escape sharp local minima
- Achieves highest **mean, median, and portfolio Sharpe Ratios** vs LSTM, ARIMA, long-only on 31 tickers
- Lower PnL variance than LSTM (4.85 vs 7.48 std across tickers)
- Supports **universality** — pooled training across assets, works on unseen stocks

### Cons
- **No transaction costs** considered
- Higher MAE/RMSE than point-estimate models (trades off accuracy for sign correctness)
- Small universe (31 tickers), short test period (Oct 2019 – Dec 2021 incl. COVID)
- GANs are inherently hard to train — sensitive to initialization, optimizer, architecture
- Computationally more expensive than LSTM/ARIMA
- No cross-asset correlation modeling (treats each series independently even in universal mode)

---

## 2. Implementation

We implement the full Fin-GAN pipeline:
1. Data download (Yahoo Finance) and preprocessing
2. ForGAN architecture (Generator + Discriminator with LSTM cells)
3. Fin-GAN loss function with all terms
4. Gradient norm matching for hyperparameter tuning
5. Training loop with validation-based loss combination selection
6. Evaluation: Sharpe Ratio, PnL, weighted strategy

In [ ]:
!pip install yfinance --quiet

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

### 2.1 Data Download & Preprocessing

We download daily OHLC data from Yahoo Finance for a set of stocks and their sector ETFs, then compute:
- Open-to-close and close-to-open log returns (alternating)
- Excess returns = stock return − sector ETF return
- Cap returns at ±15%
- Create sliding windows: condition window L=10 (5 trading days), target = next return

In [ ]:
# Define universe — a subset of what the paper uses
STOCK_ETF_MAP = {
    'PFE': 'XLV',   # Health Care
    'KO': 'XLP',    # Consumer Staples
    'GS': 'XLF',    # Financials
    'AMZN': 'XLY',  # Consumer Discretionary
    'IBM': 'XLK',   # Technology
}

ALL_TICKERS = list(STOCK_ETF_MAP.keys()) + list(set(STOCK_ETF_MAP.values()))
print(f'Downloading: {ALL_TICKERS}')

# Download data
data = yf.download(ALL_TICKERS, start='2005-01-01', end='2024-01-01', auto_adjust=False)
print(f'\nDownloaded shape: {data.shape}')
print(f'Date range: {data.index[0].date()} to {data.index[-1].date()}')

In [ ]:
def compute_oc_co_returns(data, ticker):
    """
    Compute alternating open-to-close and close-to-open log returns.
    Returns a 1D series alternating: OC_day1, CO_day1->day2, OC_day2, CO_day2->day3, ...
    """
    open_px = data['Open'][ticker].dropna()
    close_px = data['Close'][ticker].dropna()
    
    # Align dates
    common_dates = open_px.index.intersection(close_px.index)
    open_px = open_px.loc[common_dates]
    close_px = close_px.loc[common_dates]
    
    # Open-to-close returns (intraday)
    oc_ret = np.log(close_px / open_px)
    
    # Close-to-open returns (overnight) — close of day t to open of day t+1
    co_ret = np.log(open_px.iloc[1:].values / close_px.iloc[:-1].values)
    co_ret = pd.Series(co_ret, index=common_dates[1:])
    
    # Interleave: OC_1, CO_1->2, OC_2, CO_2->3, ...
    # For n days, we get n OC returns and (n-1) CO returns
    # Pair them: (OC_1, CO_1->2), (OC_2, CO_2->3), ..., last OC has no pair
    combined = []
    dates_combined = []
    for i in range(len(co_ret)):
        combined.append(oc_ret.iloc[i])
        dates_combined.append(common_dates[i])
        combined.append(co_ret.iloc[i])
        dates_combined.append(common_dates[i])  # same date for CO
    # Add last OC
    combined.append(oc_ret.iloc[-1])
    dates_combined.append(common_dates[-1])
    
    return np.array(combined)


def compute_excess_returns(stock_returns, etf_returns):
    """Excess return = stock return - ETF return. Requires same length."""
    min_len = min(len(stock_returns), len(etf_returns))
    return stock_returns[:min_len] - etf_returns[:min_len]


def cap_returns(returns, cap=0.15):
    """Cap returns at ±cap as in the paper."""
    return np.clip(returns, -cap, cap)


def create_windows(returns, L=10):
    """
    Create sliding windows: each row is [x_{t-(L-1)}, ..., x_t, x_{t+1}]
    First L columns = condition, last column = target.
    """
    n = len(returns)
    if n <= L:
        return np.array([])
    windows = np.lib.stride_tricks.sliding_window_view(returns, L + 1)
    return windows  # shape: (n - L, L + 1)


# Process one stock for demonstration
DEMO_STOCK = 'PFE'
DEMO_ETF = STOCK_ETF_MAP[DEMO_STOCK]
L = 10  # condition window (5 days × 2 returns per day)

stock_rets = compute_oc_co_returns(data, DEMO_STOCK)
etf_rets = compute_oc_co_returns(data, DEMO_ETF)
excess_rets = compute_excess_returns(stock_rets, etf_rets)
excess_rets = cap_returns(excess_rets)

print(f'{DEMO_STOCK} excess returns: {len(excess_rets)} observations')
print(f'Mean: {excess_rets.mean():.6f}, Std: {excess_rets.std():.6f}')

windows = create_windows(excess_rets, L=L)
print(f'Windows shape: {windows.shape}  (samples × (condition + target))')

In [ ]:
# Train / Validation / Test split: 80-10-10 chronological
n = len(windows)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
n_test = n - n_train - n_val

train_data = windows[:n_train]
val_data = windows[n_train:n_train + n_val]
test_data = windows[n_train + n_val:]

print(f'Train: {train_data.shape}, Val: {val_data.shape}, Test: {test_data.shape}')

# Quick visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(excess_rets[:2000], linewidth=0.5)
axes[0].set_title(f'{DEMO_STOCK} Excess Returns (first 2000 obs)')
axes[0].set_ylabel('Return')
axes[1].hist(excess_rets, bins=100, edgecolor='black', linewidth=0.3)
axes[1].set_title('Distribution of Excess Returns')
axes[1].set_xlabel('Return')
plt.tight_layout()
plt.show()

### 2.2 PyTorch Dataset

In [ ]:
class ReturnDataset(Dataset):
    """Dataset of (condition_window, target) pairs."""
    def __init__(self, windows):
        self.conditions = torch.FloatTensor(windows[:, :-1])  # (N, L)
        self.targets = torch.FloatTensor(windows[:, -1])       # (N,)
    
    def __len__(self):
        return len(self.targets)
    
    def __getitem__(self, idx):
        return self.conditions[idx], self.targets[idx]


BATCH_SIZE = 100

train_dataset = ReturnDataset(train_data)
val_dataset = ReturnDataset(val_data)
test_dataset = ReturnDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')

### 2.3 ForGAN Architecture

The **Generator** takes:
- Condition window (L past returns) → LSTM → hidden representation $h_{RG}$
- Noise vector $z \sim \mathcal{N}(0, I)$ of dimension N
- Concatenate $[h_{RG}, z]$ → Dense(RG+N) → Dense(1) → predicted return $\tilde{x}_{t+1}$

The **Discriminator** takes:
- Condition window + target (real or generated) as a sequence of L+1 values → LSTM → hidden $h_{RD}$
- Dense(1) → sigmoid → probability of being real

In [ ]:
class Generator(nn.Module):
    """
    ForGAN Generator with LSTM for condition representation learning.
    
    Architecture:
        condition (L,) -> LSTM(RG) -> h_RG
        [h_RG ; z] -> Dense(RG+N) -> ReLU -> Dense(1) -> x_hat
    """
    def __init__(self, condition_len=10, noise_dim=8, hidden_dim=8):
        super().__init__()
        self.noise_dim = noise_dim
        self.hidden_dim = hidden_dim
        
        # LSTM for condition representation
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, batch_first=True)
        
        # Dense layers after concatenation
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim + noise_dim, hidden_dim + noise_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim + noise_dim, 1)
        )
        
        # Xavier initialization
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                nn.init.xavier_normal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
    
    def forward(self, condition, noise):
        """
        Args:
            condition: (batch, L) - past L returns
            noise: (batch, noise_dim) - Gaussian noise
        Returns:
            x_hat: (batch,) - predicted return
        """
        # Reshape condition for LSTM: (batch, L, 1)
        cond_seq = condition.unsqueeze(-1)
        _, (h_n, _) = self.lstm(cond_seq)  # h_n: (1, batch, hidden_dim)
        h = h_n.squeeze(0)  # (batch, hidden_dim)
        
        # Concatenate with noise
        combined = torch.cat([h, noise], dim=1)  # (batch, hidden_dim + noise_dim)
        
        # Generate
        x_hat = self.fc(combined).squeeze(-1)  # (batch,)
        return x_hat


class Discriminator(nn.Module):
    """
    ForGAN Discriminator with LSTM for series representation learning.
    
    Architecture:
        [condition ; target_or_generated] (L+1,) -> LSTM(RD) -> h_RD
        h_RD -> Dense(1) -> Sigmoid -> probability of real
    """
    def __init__(self, condition_len=10, hidden_dim=8):
        super().__init__()
        
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                nn.init.xavier_normal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
    
    def forward(self, condition, value):
        """
        Args:
            condition: (batch, L)
            value: (batch,) - real target or generated
        Returns:
            prob: (batch,) - probability of input being real
        """
        # Concatenate condition with value: (batch, L+1, 1)
        seq = torch.cat([condition, value.unsqueeze(1)], dim=1).unsqueeze(-1)
        _, (h_n, _) = self.lstm(seq)
        h = h_n.squeeze(0)  # (batch, hidden_dim)
        prob = self.fc(h).squeeze(-1)  # (batch,)
        return prob


# Instantiate
NOISE_DIM = 8
HIDDEN_DIM = 8

gen = Generator(condition_len=L, noise_dim=NOISE_DIM, hidden_dim=HIDDEN_DIM).to(device)
disc = Discriminator(condition_len=L, hidden_dim=HIDDEN_DIM).to(device)

print(f'Generator params: {sum(p.numel() for p in gen.parameters()):,}')
print(f'Discriminator params: {sum(p.numel() for p in disc.parameters()):,}')

### 2.4 Fin-GAN Loss Function Terms

We implement all four economics-driven loss terms:

1. **PnL\***: smooth approximation to PnL using $\tanh(k \hat{x}) \cdot x$
2. **MSE**: standard mean squared error
3. **SR\***: differentiable Sharpe ratio of approximate PnLs within the minibatch
4. **STD**: standard deviation of approximate PnLs

In [ ]:
K_TANH = 100  # Controls tanh approximation sharpness


def pnl_star_per_sample(x_real, x_hat):
    """Smooth PnL per sample: tanh(k * x_hat) * x_real."""
    return torch.tanh(K_TANH * x_hat) * x_real


def loss_pnl(x_real, x_hat):
    """PnL* term — mean smooth PnL (to be MAXIMIZED, so we return negative for minimization)."""
    return pnl_star_per_sample(x_real, x_hat).mean()


def loss_mse(x_real, x_hat):
    """MSE term (to be MINIMIZED)."""
    return ((x_real - x_hat) ** 2).mean()


def loss_sr(x_real, x_hat):
    """SR* term — differentiable Sharpe ratio of approximate PnLs (to be MAXIMIZED)."""
    pnls = pnl_star_per_sample(x_real, x_hat)
    mean_pnl = pnls.mean()
    std_pnl = pnls.std()
    if std_pnl < 1e-8:
        return torch.tensor(0.0, device=x_real.device)
    return mean_pnl / std_pnl


def loss_std(x_real, x_hat):
    """STD term — std of approximate PnLs (to be MINIMIZED)."""
    pnls = pnl_star_per_sample(x_real, x_hat)
    return pnls.std()


# BCE loss for generator: -E[log(D(G(z)))]
bce_loss = nn.BCELoss()

print('Loss functions defined.')

### 2.5 Gradient Norm Matching

The paper avoids manual hyperparameter tuning by running 25 epochs with BCE loss only, recording gradient norms of all loss terms w.r.t. generator parameters, and setting each coefficient as the mean ratio of the BCE gradient norm to that term's gradient norm.

In [ ]:
def compute_grad_norm(loss, model):
    """Compute the L2 norm of gradients of `loss` w.r.t. model parameters."""
    grads = torch.autograd.grad(loss, model.parameters(), retain_graph=True, allow_unused=True)
    total_norm = 0.0
    for g in grads:
        if g is not None:
            total_norm += g.norm(2).item() ** 2
    return total_norm ** 0.5


def gradient_norm_matching(gen, disc, train_loader, n_epochs=25, lr=1e-4):
    """
    Step 1 of Algorithm 1: Train with BCE only for n_epochs,
    compute gradient norms of all loss terms to determine α, β, γ, δ.
    """
    opt_g = optim.RMSprop(gen.parameters(), lr=lr)
    opt_d = optim.RMSprop(disc.parameters(), lr=lr)
    
    ratios = {'alpha': [], 'beta': [], 'gamma': [], 'delta': []}
    
    for epoch in range(n_epochs):
        for cond, target in train_loader:
            cond, target = cond.to(device), target.to(device)
            bs = cond.size(0)
            noise = torch.randn(bs, NOISE_DIM, device=device)
            
            # --- Update discriminator ---
            x_hat = gen(cond, noise).detach()
            real_labels = torch.ones(bs, device=device) * 0.9  # label smoothing
            fake_labels = torch.zeros(bs, device=device) + 0.1
            
            d_real = disc(cond, target)
            d_fake = disc(cond, x_hat)
            d_loss = (bce_loss(d_real, real_labels) + bce_loss(d_fake, fake_labels)) / 2
            
            opt_d.zero_grad()
            d_loss.backward()
            opt_d.step()
            
            # --- Compute generator gradient norms ---
            noise = torch.randn(bs, NOISE_DIM, device=device)
            x_hat = gen(cond, noise)
            d_gen = disc(cond, x_hat)
            
            g_bce = bce_loss(d_gen, torch.ones(bs, device=device))
            g_pnl = loss_pnl(target, x_hat)
            g_mse = loss_mse(target, x_hat)
            g_sr = loss_sr(target, x_hat)
            g_std = loss_std(target, x_hat)
            
            gn_bce = compute_grad_norm(g_bce, gen)
            gn_pnl = compute_grad_norm(g_pnl, gen)
            gn_mse = compute_grad_norm(g_mse, gen)
            gn_sr = compute_grad_norm(g_sr, gen)
            gn_std = compute_grad_norm(g_std, gen)
            
            if gn_pnl > 1e-10:
                ratios['alpha'].append(gn_bce / gn_pnl)
            if gn_mse > 1e-10:
                ratios['beta'].append(gn_bce / gn_mse)
            if gn_sr > 1e-10:
                ratios['gamma'].append(gn_bce / gn_sr)
            if gn_std > 1e-10:
                ratios['delta'].append(gn_bce / gn_std)
            
            # Update generator with BCE only
            opt_g.zero_grad()
            g_bce.backward()
            opt_g.step()
    
    alpha = np.mean(ratios['alpha']) if ratios['alpha'] else 1.0
    beta = np.mean(ratios['beta']) if ratios['beta'] else 1.0
    gamma = np.mean(ratios['gamma']) if ratios['gamma'] else 1.0
    delta = np.mean(ratios['delta']) if ratios['delta'] else 1.0
    
    print(f'Gradient norm matching results:')
    print(f'  α (PnL):   {alpha:.4f}')
    print(f'  β (MSE):   {beta:.4f}')
    print(f'  γ (SR):    {gamma:.4f}')
    print(f'  δ (STD):   {delta:.4f}')
    
    return alpha, beta, gamma, delta


alpha, beta, gamma, delta = gradient_norm_matching(gen, disc, train_loader, n_epochs=25)

### 2.6 Loss Combinations

The paper considers 8 valid Fin-GAN loss combinations. For each, we train a copy of the generator from the post-gradient-matching state, then select the combination with the best validation Sharpe Ratio.

In [ ]:
# Define the 8 Fin-GAN loss combinations
LOSS_COMBOS = {
    'PnL':          {'alpha': True, 'beta': False, 'gamma': False, 'delta': False},
    'SR':           {'alpha': False, 'beta': False, 'gamma': True,  'delta': False},
    'PnL_MSE':      {'alpha': True, 'beta': True,  'gamma': False, 'delta': False},
    'PnL_SR':       {'alpha': True, 'beta': False, 'gamma': True,  'delta': False},
    'SR_MSE':       {'alpha': False, 'beta': True,  'gamma': True,  'delta': False},
    'PnL_MSE_SR':   {'alpha': True, 'beta': True,  'gamma': True,  'delta': False},
    'PnL_STD':      {'alpha': True, 'beta': False, 'gamma': False, 'delta': True},
    'PnL_STD_MSE':  {'alpha': True, 'beta': True,  'gamma': False, 'delta': True},
}


def fin_gan_generator_loss(x_real, x_hat, d_output, combo, alpha, beta, gamma, delta):
    """
    Compute the Fin-GAN generator loss:
    L_G = BCE - α*PnL* + β*MSE - γ*SR* + δ*STD
    """
    bs = x_real.size(0)
    loss = bce_loss(d_output, torch.ones(bs, device=x_real.device))
    
    if combo['alpha']:
        loss = loss - alpha * loss_pnl(x_real, x_hat)
    if combo['beta']:
        loss = loss + beta * loss_mse(x_real, x_hat)
    if combo['gamma']:
        loss = loss - gamma * loss_sr(x_real, x_hat)
    if combo['delta']:
        loss = loss + delta * loss_std(x_real, x_hat)
    
    return loss


print(f'Defined {len(LOSS_COMBOS)} loss combinations: {list(LOSS_COMBOS.keys())}')

### 2.7 Evaluation Metrics: Weighted Strategy & Sharpe Ratio

For each condition in the val/test set, we draw B=1000 noise samples from the generator. The probability of a positive forecast determines trade size. We compute the annualized Sharpe ratio of the resulting daily PnL series.

In [ ]:
B_SAMPLES = 500  # number of Monte Carlo samples per condition (paper uses 1000; we use 500 for speed)


@torch.no_grad()
def evaluate_sharpe(gen, dataset, B=B_SAMPLES):
    """
    Evaluate annualized Sharpe Ratio using the weighted strategy.
    
    For each condition, draw B noise samples, compute p_u (prob of positive forecast)
    and p_d (prob of negative forecast). Trade weight = p_u - p_d.
    Pair up consecutive half-day returns into daily PnLs.
    """
    gen.eval()
    conditions = dataset.conditions.to(device)
    targets = dataset.targets.numpy()
    n = len(targets)
    
    # Generate B samples for each condition
    all_weights = np.zeros(n)
    all_means = np.zeros(n)
    
    for i in range(n):
        cond = conditions[i].unsqueeze(0).expand(B, -1)  # (B, L)
        noise = torch.randn(B, NOISE_DIM, device=device)
        samples = gen(cond, noise).cpu().numpy()  # (B,)
        
        p_u = (samples >= 0).mean()
        p_d = (samples < 0).mean()
        all_weights[i] = p_u - p_d
        all_means[i] = samples.mean()
    
    # Weighted PnL per half-day (in bps)
    wpnl_half = 10000 * all_weights * targets
    
    # Pair up into daily PnLs
    n_days = n // 2
    daily_pnl = np.array([wpnl_half[2*i] + wpnl_half[2*i+1] for i in range(n_days)])
    
    mean_pnl = daily_pnl.mean()
    std_pnl = daily_pnl.std()
    
    if std_pnl < 1e-10:
        sr = 0.0
    else:
        sr = np.sqrt(252) * mean_pnl / std_pnl
    
    gen.train()
    return sr, mean_pnl, daily_pnl, all_means


print('Evaluation functions defined.')

### 2.8 Training Loop

We train each loss combination for 100 epochs (paper uses 100), then select the best based on validation SR.

In [ ]:
import copy

N_EPOCHS_TRAIN = 100
LR = 1e-4
MODE_COLLAPSE_THRESH = 0.0002


def train_fin_gan(gen_init, disc_init, combo_name, combo, alpha, beta, gamma, delta,
                  train_loader, val_dataset, n_epochs=N_EPOCHS_TRAIN, lr=LR):
    """
    Train a Fin-GAN with a specific loss combination.
    Returns the trained generator and validation Sharpe Ratio.
    """
    # Deep copy the models from the gradient-matching state
    gen_copy = copy.deepcopy(gen_init).to(device)
    disc_copy = copy.deepcopy(disc_init).to(device)
    
    opt_g = optim.RMSprop(gen_copy.parameters(), lr=lr)
    opt_d = optim.RMSprop(disc_copy.parameters(), lr=lr)
    
    for epoch in range(n_epochs):
        for cond, target in train_loader:
            cond, target = cond.to(device), target.to(device)
            bs = cond.size(0)
            
            # --- Discriminator update ---
            noise = torch.randn(bs, NOISE_DIM, device=device)
            x_hat = gen_copy(cond, noise).detach()
            
            d_real = disc_copy(cond, target)
            d_fake = disc_copy(cond, x_hat)
            d_loss = (bce_loss(d_real, torch.ones(bs, device=device) * 0.9) +
                      bce_loss(d_fake, torch.zeros(bs, device=device) + 0.1)) / 2
            
            opt_d.zero_grad()
            d_loss.backward()
            opt_d.step()
            
            # --- Generator update ---
            noise = torch.randn(bs, NOISE_DIM, device=device)
            x_hat = gen_copy(cond, noise)
            d_gen = disc_copy(cond, x_hat)
            
            g_loss = fin_gan_generator_loss(target, x_hat, d_gen, combo,
                                            alpha, beta, gamma, delta)
            
            opt_g.zero_grad()
            g_loss.backward()
            opt_g.step()
    
    # Evaluate on validation set
    val_sr, val_pnl, _, _ = evaluate_sharpe(gen_copy, val_dataset, B=200)
    
    # Check for mode collapse
    gen_copy.eval()
    test_cond = val_dataset.conditions[:1].to(device).expand(100, -1)
    test_noise = torch.randn(100, NOISE_DIM, device=device)
    with torch.no_grad():
        test_out = gen_copy(test_cond, test_noise).cpu().numpy()
    mode_collapse = test_out.std() < MODE_COLLAPSE_THRESH
    gen_copy.train()
    
    return gen_copy, val_sr, val_pnl, mode_collapse


print('Training function defined. Starting training...')

In [ ]:
# Train all 8 loss combinations and select the best
results = {}

for name, combo in LOSS_COMBOS.items():
    print(f'\nTraining combo: {name}...')
    trained_gen, val_sr, val_pnl, mc = train_fin_gan(
        gen, disc, name, combo, alpha, beta, gamma, delta,
        train_loader, val_dataset
    )
    results[name] = {
        'gen': trained_gen,
        'val_sr': val_sr,
        'val_pnl': val_pnl,
        'mode_collapse': mc
    }
    mc_str = ' [MODE COLLAPSE]' if mc else ''
    print(f'  Val SR: {val_sr:.4f}, Val PnL: {val_pnl:.2f} bps{mc_str}')

# Select best (excluding mode-collapsed)
valid_results = {k: v for k, v in results.items() if not v['mode_collapse']}
if not valid_results:
    valid_results = results  # fallback

best_name = max(valid_results, key=lambda k: valid_results[k]['val_sr'])
best_gen = valid_results[best_name]['gen']

print(f'\n=== Best loss combination: {best_name} (Val SR: {valid_results[best_name]["val_sr"]:.4f}) ===')

### 2.9 Test Set Evaluation

In [ ]:
# Evaluate on test set
test_sr, test_pnl, daily_pnl, test_means = evaluate_sharpe(best_gen, test_dataset, B=B_SAMPLES)

# Also compute MAE and RMSE using point estimates (mean of generated distribution)
test_targets = test_dataset.targets.numpy()
mae = np.abs(test_targets - test_means).mean()
rmse = np.sqrt(((test_targets - test_means) ** 2).mean())

print(f'=== Fin-GAN Test Results ({DEMO_STOCK}, combo: {best_name}) ===')
print(f'Annualized Sharpe Ratio: {test_sr:.4f}')
print(f'Mean Daily PnL (bps):    {test_pnl:.2f}')
print(f'MAE:                     {mae:.6f}')
print(f'RMSE:                    {rmse:.6f}')

In [ ]:
# Cumulative PnL plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(np.cumsum(daily_pnl), linewidth=1)
axes[0].set_title(f'Fin-GAN Cumulative PnL — {DEMO_STOCK} (Test Set)')
axes[0].set_xlabel('Trading Day')
axes[0].set_ylabel('Cumulative PnL (bps)')
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[0].grid(True, alpha=0.3)

# Distribution of generated forecasts for a sample condition
best_gen.eval()
sample_idx = len(test_dataset) // 2
sample_cond = test_dataset.conditions[sample_idx:sample_idx+1].to(device).expand(1000, -1)
sample_noise = torch.randn(1000, NOISE_DIM, device=device)
with torch.no_grad():
    sample_forecasts = best_gen(sample_cond, sample_noise).cpu().numpy()
best_gen.train()

true_val = test_dataset.targets[sample_idx].item()
axes[1].hist(sample_forecasts, bins=50, alpha=0.7, edgecolor='black', linewidth=0.3, density=True)
axes[1].axvline(true_val, color='red', linewidth=2, label=f'True value: {true_val:.4f}')
axes[1].axvline(0, color='gray', linestyle='--', linewidth=0.5)
axes[1].set_title(f'Generated Distribution for Sample Condition')
axes[1].set_xlabel('Forecasted Return')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.10 Baseline Comparison: LSTM

We train a simple LSTM regressor (as in the paper) for comparison.

In [ ]:
class LSTMForecaster(nn.Module):
    """Simple LSTM for point-estimate forecasting."""
    def __init__(self, hidden_dim=8):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                nn.init.xavier_normal_(param)
    
    def forward(self, x):
        # x: (batch, L)
        x = x.unsqueeze(-1)  # (batch, L, 1)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n.squeeze(0)).squeeze(-1)


# Train LSTM
lstm_model = LSTMForecaster(hidden_dim=HIDDEN_DIM).to(device)
lstm_opt = optim.RMSprop(lstm_model.parameters(), lr=LR)
mse_criterion = nn.MSELoss()

for epoch in range(125):
    for cond, target in train_loader:
        cond, target = cond.to(device), target.to(device)
        pred = lstm_model(cond)
        loss = mse_criterion(pred, target)
        lstm_opt.zero_grad()
        loss.backward()
        lstm_opt.step()

# Evaluate LSTM
lstm_model.eval()
with torch.no_grad():
    lstm_preds = lstm_model(test_dataset.conditions.to(device)).cpu().numpy()

lstm_targets = test_dataset.targets.numpy()
lstm_signs = np.sign(lstm_preds)
lstm_signs[lstm_signs == 0] = 1

# Unweighted PnL
lstm_pnl_half = 10000 * lstm_signs * lstm_targets
n_days = len(lstm_targets) // 2
lstm_daily_pnl = np.array([lstm_pnl_half[2*i] + lstm_pnl_half[2*i+1] for i in range(n_days)])
lstm_mean_pnl = lstm_daily_pnl.mean()
lstm_std_pnl = lstm_daily_pnl.std()
lstm_sr = np.sqrt(252) * lstm_mean_pnl / lstm_std_pnl if lstm_std_pnl > 1e-10 else 0

lstm_mae = np.abs(lstm_targets - lstm_preds).mean()
lstm_rmse = np.sqrt(((lstm_targets - lstm_preds) ** 2).mean())

print(f'=== LSTM Baseline Test Results ===')
print(f'Annualized Sharpe Ratio: {lstm_sr:.4f}')
print(f'Mean Daily PnL (bps):    {lstm_mean_pnl:.2f}')
print(f'MAE:                     {lstm_mae:.6f}')
print(f'RMSE:                    {lstm_rmse:.6f}')

In [ ]:
# Long-only baseline
longonly_pnl_half = 10000 * lstm_targets  # sign always +1
lo_daily_pnl = np.array([longonly_pnl_half[2*i] + longonly_pnl_half[2*i+1] for i in range(n_days)])
lo_sr = np.sqrt(252) * lo_daily_pnl.mean() / lo_daily_pnl.std() if lo_daily_pnl.std() > 1e-10 else 0

print(f'=== Long-Only Baseline ===')
print(f'Annualized Sharpe Ratio: {lo_sr:.4f}')
print(f'Mean Daily PnL (bps):    {lo_daily_pnl.mean():.2f}')

In [ ]:
# Comparison summary
print(f'\n{"="*60}')
print(f'{"Model":<20} {"SR":>10} {"PnL (bps)":>12} {"MAE":>10} {"RMSE":>10}')
print(f'{"="*60}')
print(f'{"Fin-GAN (" + best_name + ")":<20} {test_sr:>10.3f} {test_pnl:>12.2f} {mae:>10.6f} {rmse:>10.6f}')
print(f'{"LSTM":<20} {lstm_sr:>10.3f} {lstm_mean_pnl:>12.2f} {lstm_mae:>10.6f} {lstm_rmse:>10.6f}')
print(f'{"Long-Only":<20} {lo_sr:>10.3f} {lo_daily_pnl.mean():>12.2f} {"N/A":>10} {"N/A":>10}')
print(f'{"="*60}')

In [ ]:
# Cumulative PnL comparison
fig, ax = plt.subplots(figsize=(12, 5))

min_days = min(len(daily_pnl), len(lstm_daily_pnl), len(lo_daily_pnl))
ax.plot(np.cumsum(daily_pnl[:min_days]), label=f'Fin-GAN ({best_name})', linewidth=1.5)
ax.plot(np.cumsum(lstm_daily_pnl[:min_days]), label='LSTM', linewidth=1.5)
ax.plot(np.cumsum(lo_daily_pnl[:min_days]), label='Long-Only', linewidth=1.5, linestyle='--')

ax.set_title(f'Cumulative PnL Comparison — {DEMO_STOCK} (Test Set)')
ax.set_xlabel('Trading Day')
ax.set_ylabel('Cumulative PnL (bps)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

### 2.11 Visualize Generated Distributions Across Loss Combinations

This replicates Figure 5 of the paper — showing how different Fin-GAN loss terms shift the generated forecast distribution for the same condition.

In [ ]:
# Generate distributions for all loss combinations for a single test condition
sample_idx = len(test_dataset) // 3
sample_cond = test_dataset.conditions[sample_idx:sample_idx+1].to(device)
true_val = test_dataset.targets[sample_idx].item()

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

for (name, res), color in zip(results.items(), colors):
    g = res['gen']
    g.eval()
    cond_expanded = sample_cond.expand(500, -1)
    noise = torch.randn(500, NOISE_DIM, device=device)
    with torch.no_grad():
        forecasts = g(cond_expanded, noise).cpu().numpy()
    g.train()
    
    mc_label = ' [MC]' if res['mode_collapse'] else ''
    ax.hist(forecasts, bins=40, alpha=0.35, label=f'{name}{mc_label}', color=color, density=True)

ax.axvline(true_val, color='black', linewidth=2, linestyle='-', label=f'True: {true_val:.4f}')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_title(f'Generated Distributions by Loss Combination (same condition)')
ax.set_xlabel('Forecasted Excess Return')
ax.set_ylabel('Density')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### 2.12 Validation SR Heatmap Across Combinations

In [ ]:
# Bar chart of validation Sharpe Ratios
names = list(results.keys())
val_srs = [results[n]['val_sr'] for n in names]
colors = ['green' if not results[n]['mode_collapse'] else 'red' for n in names]
best_idx = names.index(best_name)
colors[best_idx] = 'gold'

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(names, val_srs, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Validation Sharpe Ratio by Loss Combination')
ax.set_ylabel('Annualized SR')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.xticks(rotation=45, ha='right')
for i, (n, v) in enumerate(zip(names, val_srs)):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print(f'\nGold = selected best | Green = valid | Red = mode collapse')

---

## 3. Summary of Results

This notebook implemented the complete Fin-GAN pipeline from Vuletić et al. (2024):

1. **Data pipeline**: Yahoo Finance → OC/CO returns → excess returns → sliding windows
2. **ForGAN architecture**: LSTM-based generator and discriminator
3. **Fin-GAN loss**: BCE + PnL* + MSE + SR* + STD terms with gradient norm matching
4. **Training**: 8 loss combinations × 100 epochs, validation-based selection
5. **Evaluation**: Weighted strategy using distributional forecasts → Sharpe Ratio

### Key Takeaways from the Paper
- The economics-driven loss terms are the main innovation — they shift generated distributions toward profitable sign prediction
- Gradient norm matching eliminates manual hyperparameter tuning for the loss coefficients
- The weighted trading strategy (using full forecast distribution) reduces PnL variance
- Fin-GAN's portfolio SR (2.107) was nearly identical to LSTM's (2.087) but with lower path volatility
- Mode collapse is effectively mitigated by the PnL term

### Caveats
- Results are sensitive to initialization and training dynamics
- We used a reduced sample count (B=500 vs 1000) and shorter training for speed
- The paper used CRSP data; we used Yahoo Finance which may differ slightly
- Transaction costs are not modeled